# Benchmark reproducible de CNN2 — CPU

Este notebook prepara el protocolo CPU. El benchmark oficial usa 10 épocas, batch 32 y tres repeticiones sin EarlyStopping. Ejecute primero el smoke test limitado; no confunda sus métricas con resultados finales.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
CHECKPOINT = ROOT / 'results/initialization/cnn2_seed42_initial.pt'
print('ROOT:', ROOT)

## 1. Precondición del entorno

La ejecución oficial debe detenerse si no usa Python 3.11.x, PyTorch 2.6.0+cpu y NumPy 1.26.4. La versión de parche objetivo es Python 3.11.9.

In [ ]:
import numpy as np
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)
assert platform.python_version() == '3.11.9'
assert str(torch.__version__) == '2.6.0+cpu'
assert np.__version__ == '1.26.4'
print('CUDA visible (permitido en benchmark CPU):', torch.cuda.is_available())

## 2. Mismo checkpoint para CPU y CUDA

No regenere pesos por dispositivo. Ambos benchmarks cargan el mismo archivo antes de cada repetición y después de descartar el warm-up.

In [ ]:
digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest().upper()
print('SHA-256:', digest)
assert digest == 'B5589BFD7B7EF9AF3E1E41ECEE5C42BA4E9BD75A6989A092963324FC4453D19F'

## 3. Metodología de tiempos

Cada batch separa: espera/carga CPU, transferencia, augmentation, zero-grad, forward, loss, backward, optimizer y métricas. `epoch_wall_s` ya contiene esos componentes: no debe sumarse nuevamente con ellos. `training_wall_s` incluye entrenamiento y validación por época; `measured_execution_s` añade restauración y test final, pero excluye warm-up.

## 4. Smoke test: una época reducida

Dos batches de entrenamiento y uno de validación/test. Las métricas no son resultados científicos.

In [ ]:
command = [
    sys.executable, '-m', 'src.benchmark',
    '--device', 'cpu', '--epochs', '1', '--batch-size', '32',
    '--repetitions', '1', '--num-workers', '0', '--seed', '42',
    '--warmup-batches', '1', '--max-train-batches', '2',
    '--max-eval-batches', '1', '--output-dir', 'results/benchmarks_smoke',
]
subprocess.run(command, cwd=ROOT, check=True)

## 5. Benchmark CPU oficial — ejecutar después

```powershell
python -m src.benchmark --device cpu --epochs 10 --batch-size 32 --repetitions 3 --num-workers 0 --seed 42 --warmup-batches 2 --output-dir results/benchmarks
```

Los resultados quedan en un directorio UTC con `benchmark.json`, `epochs.csv`, `runs.csv` y `benchmark.log`. Ajuste `num-workers` solo mediante un experimento documentado y use el mismo valor para CPU/CUDA.